# World Cup Sentiment Tracker

## Data Exploration **(Real Tweets)** 

---

### First Dataset:

Source type:  
Replies/comments under official football tweets  
(less likely to be biased and/or noisy since official posts are usually neutral/promotional, but the replies are fan reactions)

Accounts:
- @FIFAWorldCup
- national team accounts
- major player/team accounts if relevant

Tweet types:
- goal announcements
- full-time result posts
- VAR/penalty/red-card posts
- lineup posts
- highlight clips

Collected unit:  
Individual reply/comment, not the official post itself

----
### Strategy:

1. Pick one match/event
2. Find official posts about that event
3. Pull replies under those posts
4. Store replies with metadata
5. Run baseline sentiment
6. Manually inspect model failures

---

## Real Football Tweet Collection

Goal:
Collect replies from official football event posts and evaluate how well the baseline sentiment model performs on real football fan language.

Questions:
1. Can we successfully collect real football discussion?
2. Does the baseline sentiment model perform reasonably well?
3. What types of football language cause model failures?

In [1]:
import sys
sys.path.append("../src")

import pandas as pd

from api.getxapi_client import (
    fetch_replies,
    normalize_replies
)

In [2]:
response = fetch_replies(tweet_id="2069468618476200189")

response.keys()

dict_keys(['tweetId', 'reply_count', 'has_more', 'next_cursor', 'replies'])

In [ ]:
rows = normalize_replies(
    response=response,
    match="Portugal vs Spain",
    event="goal",
    source_account="FIFAWorldCup",
    team="Portugal"
)

df = pd.DataFrame(rows)

df.head()

,tweet_id,parent_tweet_id,url,timestamp,text,author_username,author_name,like_count,reply_count,view_count,match,team,player,event,source_account,source
0,2069547886975635584,2069468618476200189,https://x.com/NoCaptionMood/status/20695478869...,Tue Jun 23 22:27:21 +0000 2026,@FIFAWorldCup https://x.com/i/status/206952949...,NoCaptionMood,No Caption Mood,64,1,18711,Portugal vs Spain,Portugal,None,goal,FIFAWorldCup,getxapi
1,2069547912825049173,2069468618476200189,https://x.com/grok/status/2069547912825049173,Tue Jun 23 22:27:27 +0000 2026,@NoCaptionMood @FIFAWorldCup Ask Grok is curre...,grok,Grok,112,1,10436,Portugal vs Spain,Portugal,None,goal,FIFAWorldCup,getxapi
2,2069468946906886206,2069468618476200189,https://x.com/Shameless_925/status/20694689469...,Tue Jun 23 17:13:40 +0000 2026,@FIFAWorldCup Stupid record. Messi has 18 WC g...,Shameless_925,Amanda❤️🌸,44,38,18993,Portugal vs Spain,Portugal,None,goal,FIFAWorldCup,getxapi
3,2069765247271850442,2069468618476200189,https://x.com/AfricaFactfile/status/2069765247...,Wed Jun 24 12:51:03 +0000 2026,@FIFAWorldCup Ronaldo Celebrating his GOALS !!...,AfricaFactfile,Africa Fact file,1,0,438,Portugal vs Spain,Portugal,None,goal,FIFAWorldCup,getxapi
4,2069760564767977758,2069468618476200189,https://x.com/sportingvibe_/status/20697605647...,Wed Jun 24 12:32:27 +0000 2026,@FIFAWorldCup Six World Cups. Six tournaments ...,sportingvibe_,Sporting Vibe,0,0,102,Portugal vs Spain,Portugal,None,goal,FIFAWorldCup,getxapi


In [4]:
df.columns

Index(['tweet_id', 'parent_tweet_id', 'url', 'timestamp', 'text',
       'author_username', 'author_name', 'like_count', 'reply_count',
       'view_count', 'match', 'team', 'player', 'event', 'source_account',
       'source'],
      dtype='str')

In [5]:
df.to_csv(
    "../data/raw/real_replies_sample.csv",
    index=False
)

In [6]:
df.shape

(38, 16)

In [7]:
df["text"].head(20)

0     @FIFAWorldCup https://x.com/i/status/206952949...
1     @NoCaptionMood @FIFAWorldCup Ask Grok is curre...
2     @FIFAWorldCup Stupid record. Messi has 18 WC g...
3     @FIFAWorldCup Ronaldo Celebrating his GOALS !!...
4     @FIFAWorldCup Six World Cups. Six tournaments ...
5     @FIFAWorldCup When Ronaldo scores\nThe whole w...
6     42 agents. 216 threads. One dashboard. Every a...
7     @FIFAWorldCup He has won how many World cup tr...
8     @FIFAWorldCup No one—I repeat, no one can ever...
9     @FIFAWorldCup Messi's fans right now🤣🤣🤣 https:...
10    @FIFAWorldCup @pmcafrica THE GREATEST OF ALL T...
11    @FIFAWorldCup He knows who he is 🔥 https://t.c...
12    @FIFAWorldCup How many total World Cup goals d...
13    @FIFAWorldCup MESSI and MBAPPE right now: http...
14    @FIFAWorldCup The only player to score in six ...
15    @FIFAWorldCup Relax… it’s just against Uzbekis...
16    With one prompt: files created, routes configu...
17    @FIFAWorldCup Another record set! https://

In [8]:
from models.sentiment_baseline import (
    load_sentiment_model,
    predict_sentiment
)

classifier = load_sentiment_model()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [9]:
results = df["text"].apply(
    lambda x: predict_sentiment(
        x,
        classifier
    )
)

In [12]:
sentiment_df = pd.DataFrame(results.tolist())

df = pd.concat(
    [
        df.reset_index(drop=True),
        sentiment_df.reset_index(drop=True)
    ],
    axis=1
)

In [13]:
sample = df.sample(
    25,
    random_state=42
)

In [14]:
sample[
    [
        "text",
        "sentiment_label",
        "sentiment_score"
    ]
]

,text,sentiment_label,sentiment_score
33,"@FIFAWorldCup From one generation to the next,...",NEGATIVE,0.523144
36,Shop Prime Day Top 100+ deals ✨,POSITIVE,0.992256
4,@FIFAWorldCup Six World Cups. Six tournaments ...,NEGATIVE,0.935591
13,@FIFAWorldCup MESSI and MBAPPE right now: http...,NEGATIVE,0.988367
30,@FIFAWorldCup https://t.co/V2tTPIsyq2,NEGATIVE,0.974734
26,@FIFAWorldCup the GREATEST🫶🏻♾️ https://t.co/Q8...,NEGATIVE,0.978479
6,42 agents. 216 threads. One dashboard. Every a...,NEGATIVE,0.975363
27,@FIFAWorldCup He will always prove doubters wr...,NEGATIVE,0.956096
24,@FIFAWorldCup @windrawwin Absolute scenes! F...,POSITIVE,0.996840
15,@FIFAWorldCup Relax… it’s just against Uzbekis...,NEGATIVE,0.576894


### Observations

The baseline sentiment model's evaluation upon this dataset is heavily skewed due to the influence of unusable tweets (media, unrelated)  

Thus the proper course of action before progressing this notebook, is to define a **data pre-processing pipeline**